# Phase 1: Single-Stream DINOv2 Spatial Baseline for Morph Attack Detection
### Single-Image Face Morphing Attack Detection (S-MAD) via Self-Supervised ViT Representations & ArcFace

**Experimental Protocol:** ISO/IEC 30107-3 Compliant Presentation Attack Detection  
**Training:** SMDD (`SMDD_release_train`) Synthetic Face Morph Database  
**Evaluation:** FRLL-Morphs Zero-Shot Generalization across 5 Generators & 3 Degradation Conditions  


## 1. Imports, Runtime Initialization & Environment Setup


In [ ]:
"""
DINO-Branch Morph Attack Detection -- Phase 1 Spatial Baseline
==============================================================
Single-file, self-contained implementation for Kaggle and Local Workstations.

Architecture (Phase 1 Baseline):
    - Backbone: DINOv2 ViT-S/14 self-supervised features (384-d)
    - Projection: Linear + BatchNorm1d (384 -> 256-d)
    - Classification: Additive Angular Margin Head (ArcFace: s=30.0, m=0.35)
    - Optimization: Two-stage schedule (Stage 1: Frozen backbone warmup, Stage 2: Differential fine-tuning)

Datasets:
    - Training & In-Domain Validation: SMDD Synthetic Morphing Face Database (train.csv)
    - Zero-Shot Out-of-Distribution Evaluation: FRLL-Morphs (5 generators x 3 degradation conditions)

Usage on Kaggle:
    1. Select Accelerator: GPU P100 or T4 x2.
    2. Upload SMDD and FRLL-Morphs datasets (or mount competition datasets).
    3. Paste this script into a code cell or upload as a notebook and run.
    4. Outputs are saved to /kaggle/working/ (best_model.pt, frll_results.csv, roc_*.png).
"""

import os
import io
import sys
import csv
import math
import json
import random
import zipfile
import argparse
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from torch.amp import autocast as _torch_autocast, GradScaler as _torch_scaler
    def autocast_ctx():
        return _torch_autocast("cuda")
    def make_scaler():
        try:
            return _torch_scaler("cuda")
        except Exception:
            return _torch_scaler()
except Exception:
    from torch.cuda.amp import autocast as _legacy_autocast, GradScaler as _legacy_scaler
    def autocast_ctx():
        return _legacy_autocast()
    def make_scaler():
        return _legacy_scaler()

import timm
from sklearn.metrics import roc_auc_score, roc_curve

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore", category=UserWarning)

# Prevent thread contention in worker pools
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"


## Configuration


In [ ]:
class Config:
    """All hyperparameters in one place."""

    # Paths -- adjust to your Kaggle dataset mount points or local directory
    SMDD_ZIP = "/kaggle/input/smdd-morph/SMDD_release_train.zip"
    FRLL_ZIP = "/kaggle/input/frll-morphs/drive-download-20261010T174433Z-1-001.zip"
    OUTPUT_DIR = "/kaggle/working"

    # Backbone
    BACKBONE = "dinov2_vits14"  # timm model name; fallback: "vit_small_patch14_dinov2.lvd142m"
    EMBED_DIM = 384             # 384 for ViT-S, 768 for ViT-B
    PROJ_DIM = 256

    # ArcFace
    ARCFACE_S = 30.0
    ARCFACE_M = 0.35
    NUM_CLASSES = 2  # 0: bonafide, 1: morph

    # Training
    IMG_SIZE = 224
    BATCH_SIZE = 64
    NUM_WORKERS = 2
    WARMUP_EPOCHS = 3
    TOTAL_EPOCHS = 10
    HEAD_LR = 1e-3
    FINETUNE_HEAD_LR = 1e-4
    FINETUNE_BACKBONE_LR = 1e-5
    WEIGHT_DECAY = 1e-4
    UNFREEZE_BLOCKS = 2  # number of last transformer blocks to unfreeze in stage 2

    # Validation split
    VAL_FRACTION = 0.2

    # Execution control
    EVAL_ONLY = False
    CKPT_TO_EVAL = "best_model.pt"

    # Reproducibility
    SEED = 42

    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


## Augmentations


In [ ]:
def get_train_transforms(img_size: int) -> A.Compose:
    """Heavy augmentation pipeline destroying compression/blur shortcuts."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        # Simulated JPEG compression (quality 20-70)
        A.ImageCompression(quality_lower=20, quality_upper=70, p=0.5),
        # Gaussian blur
        A.GaussianBlur(blur_limit=(3, 7), sigma_limit=(0.1, 2.0), p=0.3),
        # Motion blur
        A.MotionBlur(blur_limit=(3, 7), p=0.2),
        # Simulated print-scan: downsample + JPEG + noise + upsample
        A.Compose([
            A.Downscale(scale_min=0.4, scale_max=0.6, p=1.0),
            A.ImageCompression(quality_lower=30, quality_upper=50, p=1.0),
            A.GaussNoise(var_limit=(5.0, 25.0), p=1.0),
        ], p=0.2),
        # Random gamma / brightness
        A.RandomGamma(gamma_limit=(70, 130), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        # Color jitter
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
        # Normalize + tensor
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size: int) -> A.Compose:
    """Deterministic eval transform."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_degraded_transforms(img_size: int, mode: str = "jpeg") -> A.Compose:
    """Apply degradation before standard eval transform for robustness slices."""
    degradation = []
    if mode == "jpeg":
        degradation.append(A.ImageCompression(quality_lower=25, quality_upper=35, p=1.0))
    elif mode == "printscan":
        degradation.extend([
            A.Downscale(scale_min=0.4, scale_max=0.6, p=1.0),
            A.ImageCompression(quality_lower=30, quality_upper=50, p=1.0),
            A.GaussNoise(var_limit=(5.0, 25.0), p=1.0),
        ])
    return A.Compose([
        *degradation,
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


## Datasets (Supports both .zip archives and extracted folders)


In [ ]:
class SMDDDataset(Dataset):
    """Reads SMDD images from a zip archive or an extracted directory using train.csv metadata."""

    def __init__(self, data_source: str, entries: list, transform=None):
        self.data_source = data_source
        self.is_zip = os.path.isfile(data_source) and zipfile.is_zipfile(data_source)
        self.entries = entries
        self.transform = transform
        self._zf = None

    def _open_zip(self):
        if self.is_zip and self._zf is None:
            self._zf = zipfile.ZipFile(self.data_source, "r")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        img_rel_path, label = self.entries[idx]
        if self.is_zip:
            self._open_zip()
            img_bytes = self._zf.read(img_rel_path)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        else:
            full_path = os.path.join(self.data_source, img_rel_path)
            img = Image.open(full_path).convert("RGB")
        img = np.array(img)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label

    def __del__(self):
        if self._zf is not None:
            try:
                self._zf.close()
            except Exception:
                pass


class FRLLDataset(Dataset):
    """Reads FRLL-Morphs images from a zip archive or an extracted directory for evaluation."""

    def __init__(self, data_source: str, bonafide_paths: list, morph_paths: list, transform=None):
        self.data_source = data_source
        self.is_zip = os.path.isfile(data_source) and zipfile.is_zipfile(data_source)
        self.entries = [(p, 0) for p in bonafide_paths] + [(p, 1) for p in morph_paths]
        self.transform = transform
        self._zf = None

    def _open_zip(self):
        if self.is_zip and self._zf is None:
            self._zf = zipfile.ZipFile(self.data_source, "r")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        img_rel_path, label = self.entries[idx]
        if self.is_zip:
            self._open_zip()
            img_bytes = self._zf.read(img_rel_path)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        else:
            full_path = os.path.join(self.data_source, img_rel_path)
            img = Image.open(full_path).convert("RGB")
        img = np.array(img)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label

    def __del__(self):
        if self._zf is not None:
            try:
                self._zf.close()
            except Exception:
                pass


def load_smdd_entries(data_source: str):
    """Parse train.csv inside the SMDD zip or directory and return list of (rel_path, label_int)."""
    if os.path.isfile(data_source) and zipfile.is_zipfile(data_source):
        with zipfile.ZipFile(data_source, "r") as zf:
            csv_bytes = zf.read("train.csv").decode("utf-8")
        reader = csv.DictReader(io.StringIO(csv_bytes))
    else:
        csv_file = os.path.join(data_source, "train.csv")
        with open(csv_file, "r", encoding="utf-8") as f:
            reader = list(csv.DictReader(f))

    entries = []
    for row in reader:
        label = 0 if row["label"] == "bonafide" else 1
        entries.append((row["image_path"], label))
    return entries


def load_frll_paths(data_source: str):
    """Enumerate FRLL-Morphs relative paths grouped by generator."""
    if os.path.isfile(data_source) and zipfile.is_zipfile(data_source):
        with zipfile.ZipFile(data_source, "r") as zf:
            all_names = zf.namelist()
    else:
        all_names = []
        for root, _, files in os.walk(data_source):
            for file in files:
                rel = os.path.relpath(os.path.join(root, file), data_source)
                all_names.append(rel.replace("\\", "/"))

    bonafide = sorted([
        n for n in all_names
        if (n.startswith("raw/") or "/raw/" in n) and n.endswith(".jpg")
    ])
    generators = {}
    for prefix in ["morph_opencv", "morph_facemorpher", "morph_webmorph", "morph_stylegan", "morph_amsl"]:
        generators[prefix] = sorted([
            n for n in all_names
            if (n.startswith(prefix + "/") or f"/{prefix}/" in n)
            and (n.endswith(".jpg") or n.endswith(".png"))
        ])
    return bonafide, generators


## Model


In [ ]:
class ArcFaceHead(nn.Module):
    """Additive angular margin classification head."""

    def __init__(self, in_features: int, num_classes: int, s: float = 30.0, m: float = 0.35):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor = None):
        w = F.normalize(self.weight, dim=1)
        x = F.normalize(embeddings, dim=1)
        cosine = F.linear(x, w)

        if labels is not None:
            theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
            one_hot = F.one_hot(labels, num_classes=cosine.size(1)).float()
            logits = torch.cos(theta + one_hot * self.m)
            logits = logits * self.s
        else:
            logits = cosine * self.s

        return logits


def create_dino_backbone(backbone_name: str, pretrained: bool = True):
    """Safely create DINOv2 backbone across timm version variants."""
    candidates = [
        backbone_name,
        "vit_small_patch14_dinov2.lvd142m",
        "dinov2_vits14",
    ]
    for name in candidates:
        try:
            return timm.create_model(name, pretrained=pretrained, num_classes=0, dynamic_img_size=True)
        except Exception:
            try:
                return timm.create_model(name, pretrained=pretrained, num_classes=0)
            except Exception:
                continue
    raise RuntimeError(f"Failed to create DINOv2 backbone from candidates: {candidates}")


class MorphDetector(nn.Module):
    """DINOv2 backbone + projection + ArcFace head (Phase 1 Baseline)."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        # Load backbone
        self.backbone = create_dino_backbone(cfg.BACKBONE, pretrained=True)

        # Projection
        self.projector = nn.Sequential(
            nn.Linear(cfg.EMBED_DIM, cfg.PROJ_DIM),
            nn.BatchNorm1d(cfg.PROJ_DIM),
        )

        # ArcFace head
        self.arc_head = ArcFaceHead(
            in_features=cfg.PROJ_DIM,
            num_classes=cfg.NUM_CLASSES,
            s=cfg.ARCFACE_S,
            m=cfg.ARCFACE_M,
        )

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_last_n_blocks(self, n: int):
        """Unfreeze the last n transformer blocks of the ViT."""
        if hasattr(self.backbone, "blocks"):
            blocks = list(self.backbone.blocks)
            for block in blocks[-n:]:
                for param in block.parameters():
                    param.requires_grad = True
        if hasattr(self.backbone, "norm"):
            for param in self.backbone.norm.parameters():
                param.requires_grad = True

    def forward(self, x, labels=None, return_scores=False):
        features = self.backbone(x)  # (B, EMBED_DIM)
        proj = self.projector(features)  # (B, PROJ_DIM)
        if return_scores:
            logits = self.arc_head(proj, labels=None)
            probs = F.softmax(logits, dim=1)
            return probs[:, 1]  # morph class probability
        logits = self.arc_head(proj, labels)  # (B, NUM_CLASSES)
        return logits

    def get_scores(self, x):
        return self.forward(x, return_scores=True)


## Metrics (ISO/IEC 30107-3)


In [ ]:
def compute_eer(labels: np.ndarray, scores: np.ndarray):
    """Compute Equal Error Rate from labels and scores."""
    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return eer, thresholds[idx]


def compute_bpcer_at_apcer(labels: np.ndarray, scores: np.ndarray, apcer_target: float):
    """
    Compute BPCER at a fixed APCER threshold.
    APCER: Attack Presentation Classification Error Rate (FNR on morph attacks).
    BPCER: Bona Fide Presentation Classification Error Rate (FPR on genuine samples).
    """
    attack_scores = scores[labels == 1]
    bonafide_scores = scores[labels == 0]

    if len(attack_scores) == 0 or len(bonafide_scores) == 0:
        return 1.0

    thresholds = np.sort(np.unique(scores))
    best_bpcer = 1.0

    for t in thresholds:
        apcer = np.mean(attack_scores < t)
        bpcer = np.mean(bonafide_scores >= t)
        if apcer <= apcer_target:
            best_bpcer = bpcer
            break

    return best_bpcer


def compute_all_metrics(labels: np.ndarray, scores: np.ndarray) -> dict:
    """Compute ISO/IEC 30107-3 metrics: EER, BPCER@APCER=1%, BPCER@APCER=5%, and ROC-AUC."""
    eer, eer_thresh = compute_eer(labels, scores)
    bpcer_1 = compute_bpcer_at_apcer(labels, scores, apcer_target=0.01)
    bpcer_5 = compute_bpcer_at_apcer(labels, scores, apcer_target=0.05)
    try:
        auc = roc_auc_score(labels, scores)
    except ValueError:
        auc = float("nan")
    return {
        "EER": eer,
        "EER_threshold": eer_thresh,
        "BPCER@APCER=1%": bpcer_1,
        "BPCER@APCER=5%": bpcer_5,
        "ROC-AUC": auc,
    }


## Training Engine


In [ ]:
def build_sampler(entries: list) -> WeightedRandomSampler:
    """Balance SMDD class ratio (5:3 bonafide to morph) during training."""
    labels = [e[1] for e in entries]
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, num_samples=len(entries), replacement=True)


def train_one_epoch(model, loader, optimizer, scaler, device, epoch):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast_ctx():
            logits = model(images, labels)
            loss = F.cross_entropy(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

        if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(loader):
            print(f"  Epoch {epoch} | Batch {batch_idx+1}/{len(loader)} | "
                  f"Loss: {loss.item():.4f} | Acc: {correct/total:.4f}")

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_labels = []
    all_scores = []
    total_val_loss = 0.0
    val_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels_dev = labels.to(device, non_blocking=True)
        with autocast_ctx():
            scores = model(images, return_scores=True)
            logits = model(images, labels=labels_dev)
            loss = F.cross_entropy(logits, labels_dev)

        total_val_loss += loss.item() * images.size(0)
        val_count += images.size(0)
        all_scores.append(scores.cpu().numpy())
        all_labels.append(labels.numpy())

    all_labels = np.concatenate(all_labels)
    all_scores = np.concatenate(all_scores)

    metrics = compute_all_metrics(all_labels, all_scores)
    metrics["val_loss"] = total_val_loss / max(1, val_count)
    return metrics, all_labels, all_scores


def train(cfg: Config):
    """Full two-stage training pipeline with multi-GPU support."""
    seed_everything(cfg.SEED)
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    num_gpus = torch.cuda.device_count()
    effective_bs = cfg.BATCH_SIZE * max(1, num_gpus)

    print("=" * 60)
    print("DINO-Branch Morph Attack Detection -- Phase 1 Spatial Baseline")
    print(f"  Available GPUs: {num_gpus} | Effective Batch Size: {effective_bs}")
    print("=" * 60)

    # ---- Data ----
    print("\n[1/5] Loading SMDD dataset...")
    all_entries = load_smdd_entries(cfg.SMDD_ZIP)
    random.shuffle(all_entries)

    val_size = int(len(all_entries) * cfg.VAL_FRACTION)
    val_entries = all_entries[:val_size]
    train_entries = all_entries[val_size:]

    train_labels = [e[1] for e in train_entries]
    val_labels = [e[1] for e in val_entries]
    print(f"  Train: {len(train_entries)} (bonafide={train_labels.count(0)}, morph={train_labels.count(1)})")
    print(f"  Val:   {len(val_entries)} (bonafide={val_labels.count(0)}, morph={val_labels.count(1)})")

    train_ds = SMDDDataset(cfg.SMDD_ZIP, train_entries, transform=get_train_transforms(cfg.IMG_SIZE))
    val_ds = SMDDDataset(cfg.SMDD_ZIP, val_entries, transform=get_val_transforms(cfg.IMG_SIZE))

    sampler = build_sampler(train_entries)
    train_loader = DataLoader(train_ds, batch_size=effective_bs, sampler=sampler,
                              num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=effective_bs, shuffle=False,
                            num_workers=cfg.NUM_WORKERS, pin_memory=True)

    # ---- Model ----
    print("\n[2/5] Building model...")
    raw_model = MorphDetector(cfg).to(cfg.DEVICE)
    raw_model.freeze_backbone()
    print(f"  Backbone: {cfg.BACKBONE} (frozen)")
    print(f"  Projection: {cfg.EMBED_DIM} -> {cfg.PROJ_DIM}")
    print(f"  ArcFace: s={cfg.ARCFACE_S}, m={cfg.ARCFACE_M}")

    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in raw_model.parameters())
    print(f"  Trainable: {trainable:,} / {total_params:,} parameters")

    model = raw_model
    if num_gpus > 1:
        print(f"  Enabling nn.DataParallel across {num_gpus} GPUs.")
        model = nn.DataParallel(raw_model)

    scaler = make_scaler()
    best_eer = 1.0
    history = []

    # ---- Stage 1: Head warm-up ----
    print("\n[3/5] Stage 1: Head warm-up...")
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, raw_model.parameters()),
        lr=cfg.HEAD_LR, weight_decay=cfg.WEIGHT_DECAY,
    )

    for epoch in range(1, cfg.WARMUP_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, cfg.DEVICE, epoch)
        val_metrics, _, _ = validate(model, val_loader, cfg.DEVICE)
        print(f"  Epoch {epoch}: loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_EER={val_metrics['EER']:.4f} val_AUC={val_metrics['ROC-AUC']:.4f}")
        history.append({"epoch": epoch, "stage": 1, "train_loss": train_loss,
                        "train_acc": train_acc, **val_metrics})

        if val_metrics["EER"] < best_eer:
            best_eer = val_metrics["EER"]
            torch.save(raw_model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "best_model.pt"))
            print(f"    -> New best EER: {best_eer:.4f}")

    # ---- Stage 2: Differential fine-tuning ----
    print(f"\n[4/5] Stage 2: Unfreezing last {cfg.UNFREEZE_BLOCKS} blocks...")
    raw_model.unfreeze_last_n_blocks(cfg.UNFREEZE_BLOCKS)

    trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
    print(f"  Trainable: {trainable:,} / {total_params:,} parameters")

    backbone_params = []
    head_params = []
    for name, param in raw_model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg.FINETUNE_BACKBONE_LR},
        {"params": head_params, "lr": cfg.FINETUNE_HEAD_LR},
    ], weight_decay=cfg.WEIGHT_DECAY)

    total_ft_epochs = cfg.TOTAL_EPOCHS - cfg.WARMUP_EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_ft_epochs))

    for epoch in range(cfg.WARMUP_EPOCHS + 1, cfg.TOTAL_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, cfg.DEVICE, epoch)
        scheduler.step()
        val_metrics, _, _ = validate(model, val_loader, cfg.DEVICE)
        print(f"  Epoch {epoch}: loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_EER={val_metrics['EER']:.4f} val_AUC={val_metrics['ROC-AUC']:.4f}")
        history.append({"epoch": epoch, "stage": 2, "train_loss": train_loss,
                        "train_acc": train_acc, **val_metrics})

        if val_metrics["EER"] < best_eer:
            best_eer = val_metrics["EER"]
            torch.save(raw_model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "best_model.pt"))
            print(f"    -> New best EER: {best_eer:.4f}")

    # Save training history
    pd.DataFrame(history).to_csv(os.path.join(cfg.OUTPUT_DIR, "training_history.csv"), index=False)
    print(f"\n  Training complete. Best validation EER: {best_eer:.4f}")

    return model


## Evaluation Engine


In [ ]:
def evaluate_frll(cfg: Config, model: nn.Module = None):
    """Evaluate checkpoint on all FRLL-Morphs generator splits + degradation conditions."""

    print("\n" + "=" * 60)
    print("Cross-Dataset Evaluation on FRLL-Morphs (ISO/IEC 30107-3)")
    print("=" * 60)

    num_gpus = torch.cuda.device_count()
    effective_bs = cfg.BATCH_SIZE * max(1, num_gpus)

    # Load model
    if model is None:
        raw_model = MorphDetector(cfg).to(cfg.DEVICE)
        ckpt_path = os.path.join(cfg.OUTPUT_DIR, cfg.CKPT_TO_EVAL)
        if not os.path.exists(ckpt_path):
            ckpt_path = cfg.CKPT_TO_EVAL
        if os.path.exists(ckpt_path):
            raw_model.load_state_dict(torch.load(ckpt_path, map_location=cfg.DEVICE))
            print(f"  Loaded checkpoint: {ckpt_path}")
        else:
            print(f"  Notice: Checkpoint {ckpt_path} not found. Running with initialized weights.")
        model = raw_model
        if num_gpus > 1:
            model = nn.DataParallel(raw_model)
    elif not isinstance(model, nn.DataParallel) and num_gpus > 1:
        model = nn.DataParallel(model)

    model.eval()

    bonafide_paths, generators = load_frll_paths(cfg.FRLL_ZIP)
    print(f"  Bonafide: {len(bonafide_paths)} images")
    for gen_name, paths in generators.items():
        print(f"  {gen_name}: {len(paths)} images")

    all_results = []
    conditions = [
        ("clean", get_val_transforms(cfg.IMG_SIZE)),
        ("jpeg_q30", get_degraded_transforms(cfg.IMG_SIZE, mode="jpeg")),
        ("printscan", get_degraded_transforms(cfg.IMG_SIZE, mode="printscan")),
    ]

    for gen_name, morph_paths in generators.items():
        if len(morph_paths) == 0:
            continue
        for cond_name, transform in conditions:
            ds = FRLLDataset(cfg.FRLL_ZIP, bonafide_paths, morph_paths, transform=transform)
            loader = DataLoader(ds, batch_size=effective_bs, shuffle=False,
                                num_workers=cfg.NUM_WORKERS, pin_memory=True)
            metrics, labels, scores = validate(model, loader, cfg.DEVICE)

            result = {"generator": gen_name, "condition": cond_name, **metrics}
            all_results.append(result)
            print(f"\n  {gen_name} [{cond_name}]:")
            print(f"    EER:             {metrics['EER']:.4f}")
            print(f"    BPCER@APCER=1%:  {metrics['BPCER@APCER=1%']:.4f}")
            print(f"    BPCER@APCER=5%:  {metrics['BPCER@APCER=5%']:.4f}")
            print(f"    ROC-AUC:         {metrics['ROC-AUC']:.4f}")

            # Honesty check / leakage warning
            if metrics["EER"] < 0.005:
                print(f"    *** NOTICE: EER near 0% -- check data splits to confirm strict zero-shot eval. ***")

            # Save ROC curve for clean condition
            if cond_name == "clean":
                fpr, tpr, _ = roc_curve(labels, scores)
                plt.figure(figsize=(6, 6))
                plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {metrics['ROC-AUC']:.4f}")
                plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
                plt.xlabel("FPR (BPCER)")
                plt.ylabel("TPR (1 - APCER)")
                plt.title(f"ROC: {gen_name} (Clean)")
                plt.legend(loc="lower right")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(os.path.join(cfg.OUTPUT_DIR, f"roc_{gen_name}.png"), dpi=150)
                plt.close()

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(os.path.join(cfg.OUTPUT_DIR, "frll_results.csv"), index=False)
    print(f"\n  Results saved to {cfg.OUTPUT_DIR}/frll_results.csv")

    print("\n" + "=" * 60)
    print("SUMMARY TABLE (ISO/IEC 30107-3)")
    print("=" * 60)
    print(results_df.to_string(index=False, float_format="%.4f"))

    return results_df


## Path Auto-Detection


In [ ]:
def auto_detect_datasets(cfg: Config):
    """Recursively search Kaggle mount points and local directories."""
    search_roots = ["/kaggle/input", "."]

    # Find SMDD
    if not os.path.exists(cfg.SMDD_ZIP):
        found = False
        for sroot in search_roots:
            if not os.path.exists(sroot):
                continue
            for root, dirs, files in os.walk(sroot):
                if "train.csv" in files:
                    cfg.SMDD_ZIP = root
                    found = True
                    break
                for f in files:
                    if f.endswith(".zip") and "smdd" in f.lower():
                        cfg.SMDD_ZIP = os.path.join(root, f)
                        found = True
                        break
                if found:
                    break
            if found:
                break

    # Find FRLL
    if not os.path.exists(cfg.FRLL_ZIP):
        found = False
        for sroot in search_roots:
            if not os.path.exists(sroot):
                continue
            for root, dirs, files in os.walk(sroot):
                if any("morph_opencv" in d for d in dirs):
                    cfg.FRLL_ZIP = root
                    found = True
                    break
                for f in files:
                    if f.endswith(".zip") and ("drive-download" in f or "frll" in f.lower()):
                        cfg.FRLL_ZIP = os.path.join(root, f)
                        found = True
                        break
                if found:
                    break
            if found:
                break


def main():
    parser = argparse.ArgumentParser(description="Phase 1 Single-Stream DINOv2 MAD")
    parser.add_argument("--eval-only", action="store_true", help="Skip training and evaluate checkpoint directly")
    parser.add_argument("--ckpt", type=str, default="best_model.pt", help="Checkpoint path to evaluate")
    args, _ = parser.parse_known_args()

    cfg = Config()
    if args.eval_only:
        cfg.EVAL_ONLY = True
    if args.ckpt:
        cfg.CKPT_TO_EVAL = args.ckpt

    auto_detect_datasets(cfg)

    print(f"SMDD path: {cfg.SMDD_ZIP} (exists={os.path.exists(cfg.SMDD_ZIP)})")
    print(f"FRLL path: {cfg.FRLL_ZIP} (exists={os.path.exists(cfg.FRLL_ZIP)})")

    if not os.path.exists(cfg.SMDD_ZIP):
        raise FileNotFoundError(f"SMDD dataset not found. Checked /kaggle/input and current directory.")
    if not os.path.exists(cfg.FRLL_ZIP):
        raise FileNotFoundError(f"FRLL dataset not found. Checked /kaggle/input and current directory.")

    model = None
    if not cfg.EVAL_ONLY:
        model = train(cfg)

    evaluate_frll(cfg, model)

    print("\n" + "=" * 60)
    print("DONE. All outputs in:", cfg.OUTPUT_DIR)
    print("=" * 60)


if __name__ == "__main__":
    main()
